In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
from src.loading_data.load_data import get_clean_2022
from src.perturbation.fitting_models import preprocess_perturbation2
from src.loading_data.data_catalogue import DataCatalogue
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

c:\Users\fergu\Documents\GitHub\london_sport2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = get_clean_2022()

['Gend3', 'Disab2_POP', 'Age9', 'Eth7', 'NSSEC5', 'Educ6', 'IMD10', 'Child4', 'WorkStat8', 'HHLiv9', 'serial', 'year', 'LCA_Class', 'MEMS7_ALL']
Data loaded with zero missing values.
Loaded primary frame...
Loaded secondary frame...
Merged Frames.
Dropped NAN values
Dropped: ['serial', 'year', 'MEMS7_ALL]


In [3]:
df.columns

Index(['LCA_Class', 'Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5',
       'IMD10', 'WorkStat8', 'Child4', 'HHLiv9', 'active_status', 'Motiva_POP',
       'motivd_POP', 'inclus_a', 'inclus_b', 'inclus_c', 'anxious', 'comm1',
       'comm2', 'happy', 'indev', 'indevtry', 'lone', 'worthw'],
      dtype='str')

In [4]:
dc = DataCatalogue()
categoricals = dc.get_perturbation_processing_categoricals()
continuous_vars = dc.get_perturbation_core_contins() + dc.get_perturbation_vars()
print(categoricals)

['active_status', 'Gend3', 'Disab2_POP', 'Eth7', 'WorkStat8', 'HHLiv9']


In [5]:
X_train, X_test, Y_train, Y_test, test_set_clusters = preprocess_perturbation2(df, 'LCA_Class', 'active_status', categoricals)

Keep columns : ['Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10', 'WorkStat8', 'Child4', 'HHLiv9', 'Motiva_POP', 'motivd_POP', 'inclus_a', 'inclus_b', 'inclus_c', 'anxious', 'comm1', 'comm2', 'happy', 'indev', 'indevtry', 'lone', 'worthw']
X_train cols:
Index(['Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10',
       'WorkStat8', 'Child4', 'HHLiv9', 'Motiva_POP', 'motivd_POP', 'inclus_a',
       'inclus_b', 'inclus_c', 'anxious', 'comm1', 'comm2', 'happy', 'indev',
       'indevtry', 'lone', 'worthw'],
      dtype='str')
X_train shape:
(2628, 23)
X_train values:
       Age9 Gend3 Eth7 Disab2_POP  Educ6  NSSEC5  IMD10 WorkStat8  Child4  \
9825    4.0     1    1          1    1.0     1.0   10.0         0     4.0   
4034    8.0     1    1          1    1.0     5.0    8.0         3     1.0   
189     4.0     1    3          1    1.0     1.0    5.0         0     2.0   
2050    4.0     2    1          1    1.0     1.0    8.0         1     2.0   
12453   3.0  

In [6]:
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
# X_test is saved as a csv during preprocess perturbations so the scaling here doesnt matter
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [7]:
model1 = LGBMClassifier(class_weight='balanced', random_state = 42)
model1.fit(X_train, Y_train)
preds = model1.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000147 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 166
[LightGBM] [Info] Number of data points in the train set: 2628, number of used features: 23
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
0.4277020027711311


In [8]:
model2 = XGBClassifier(random_state = 42)
model2.fit(X_train, Y_train)
preds = model2.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)


0.3928951952710052


In [9]:
model3 = RandomForestClassifier(random_state=42, class_weight='balanced')
model3.fit(X_train, Y_train)
preds = model3.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

0.4109097430775239


In [ ]:
model4 = CatBoostClassifier(random_state = 42, auto_class_weights='Balanced', iterations=30) # auto_class_weights='balanced'
model4.fit(X_train, Y_train, cat_features=['Gend3', 'Disab2_POP', 'Eth7', 'WorkStat8', 'HHLiv9'])
preds = model4.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

Learning rate set to 0.5
0:	learn: 1.0691646	total: 35.1ms	remaining: 1.02s
1:	learn: 1.0459987	total: 112ms	remaining: 1.57s
2:	learn: 1.0049598	total: 189ms	remaining: 1.7s
3:	learn: 0.9903511	total: 258ms	remaining: 1.68s
4:	learn: 0.9681550	total: 318ms	remaining: 1.59s
5:	learn: 0.9531023	total: 376ms	remaining: 1.5s
6:	learn: 0.9291308	total: 435ms	remaining: 1.43s
7:	learn: 0.9205693	total: 493ms	remaining: 1.35s
8:	learn: 0.9191092	total: 522ms	remaining: 1.22s
9:	learn: 0.9051826	total: 580ms	remaining: 1.16s
10:	learn: 0.9005476	total: 636ms	remaining: 1.1s
11:	learn: 0.8898670	total: 695ms	remaining: 1.04s
12:	learn: 0.8804586	total: 757ms	remaining: 990ms
13:	learn: 0.8746528	total: 816ms	remaining: 932ms
14:	learn: 0.8610954	total: 883ms	remaining: 883ms
15:	learn: 0.8548774	total: 939ms	remaining: 821ms
16:	learn: 0.8460558	total: 996ms	remaining: 762ms
17:	learn: 0.8356164	total: 1.05s	remaining: 702ms
18:	learn: 0.8257835	total: 1.11s	remaining: 640ms
19:	learn: 0.82298